# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MEDOXIII/FlyRank-Internship-ML/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/MEDOXIII/FlyRank-Internship-ML"
REPO_DIR = "FlyRank-Internship-ML"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # find the repo root from wherever this kernel started
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found"

Working dir: D:\Projects\FlyRank-Internship-ML\FlyRank-Internship-ML


## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

I chose the Refresh / Content Opportunity Scoring lane. My goal is to identify which content pages should be reviewed first based on observable search and engagement signals. I selected this lane because it focuses on a practical business problem, supports both ranking and machine learning approaches, and provides clear evaluation metrics. It also aligns well with the starter dataset and offers a straightforward path from exploratory analysis to actionable recommendations.


In [2]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(f"loaded {df.shape[0]:,} rows x {df.shape[1]} cols")

lane_columns = ["trend_direction", "avg_position", "ctr", "days_since_last_update", "impressions_90d"]
print("lane-relevant columns present:", all(c in df.columns for c in lane_columns))

loaded 30,000 rows x 44 cols
lane-relevant columns present: True


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

This project improves the decision of which pages should be reviewed and prioritized for content updates. The primary users would be SEO specialists, content managers, or growth teams responsible for maintaining content performance. A wrong recommendation could waste limited review time on pages with little improvement potential, while more important opportunities may be overlooked. However, the system is intended as a decision-support tool rather than an automated decision maker.


In [3]:
print("distinct clients (each with its own review queue):", df["client_id"].nunique())

pages_per_client = df.groupby("client_id").size()
print("pages per client - min/median/max:",
      pages_per_client.min(), pages_per_client.median(), pages_per_client.max())

distinct clients (each with its own review queue): 32
pages per client - min/median/max: 3 567.0 7008


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

The starter dataset contains several strong indicators that make the Refresh / Content Opportunity Scoring lane worth pursuing.

First, 8,031 pages show a declining trend while still receiving at least 1,000 impressions in the last 90 days. This suggests that a large number of pages continue to attract meaningful search demand despite signs of performance deterioration, making them strong refresh candidates.

Second, 5,178 pages rank on the first page of search results (average position ≤ 10) yet have a CTR below 0.5% while still receiving at least 1,000 impressions in the last 90 days. These pages already have visibility and real search demand but are failing to convert impressions into clicks, indicating a significant optimization opportunity.

Third, 5,479 pages have not been updated for at least 90 days while still receiving more than 1,000 impressions. This shows that substantial traffic is being directed to aging content, creating an opportunity to prioritize refresh efforts where audience demand already exists.

Together, these findings demonstrate that the dataset contains thousands of potentially actionable pages, making a content opportunity scoring system a valuable decision-support tool for prioritizing content reviews and refreshes.


In [4]:
claim1 = ((df["trend_direction"] == "down") & (df["impressions_90d"] >= 1000)).sum()
print("declining pages with >=1,000 impressions_90d:", claim1)

claim2 = ((df["avg_position"] > 0) & (df["avg_position"] <= 10)
          & (df["ctr"] < 0.5) & (df["impressions_90d"] >= 1000)).sum()
print("page-1 pages (avg_position 1-10, real data) with ctr<0.5% and >=1,000 impressions_90d:", claim2)

claim3 = ((df["days_since_last_update"] >= 90) & (df["impressions_90d"] > 1000)).sum()
print("stale pages (>=90d since update) with >1,000 impressions_90d:", claim3)

declining pages with >=1,000 impressions_90d: 8031
page-1 pages (avg_position 1-10, real data) with ctr<0.5% and >=1,000 impressions_90d: 5178
stale pages (>=90d since update) with >1,000 impressions_90d: 5479


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

This project will provide observed and directional evidence about which pages appear to have higher refresh or review priority based on historical search and engagement signals. It can support decisions by ranking pages according to estimated opportunity or decline risk. However, it will not provide causal proof that updating a page will improve performance, and it will not claim to predict Google’s algorithms or ranking factors. Any findings should be interpreted as decision-support evidence rather than guarantees of future outcomes.


In [5]:
experiment_like = [c for c in df.columns
                   if any(k in c.lower() for k in ["variant", "test_group", "experiment", "treatment", "ab_"])]
print("experiment/treatment columns in this dataset:", experiment_like or "none")
print("-> no randomized intervention structure exists here, so no causal claim can be backed by this data")

experiment/treatment columns in this dataset: none
-> no randomized intervention structure exists here, so no causal claim can be backed by this data


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.